<a href="https://colab.research.google.com/github/CreatorDave/201lab05a/blob/main/LAB_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##PART 1

- You're the data scientist for a BPO company where employees use a portal for taking calls and ticketing. You are given the file 'user_id' consisting of the employee ids.
<br>
- The company is planning to change the layout and has a proposed New Version
<br>
- Currently, there are 1000 employees.

##Question 1: Your boss says that it takes money to deploy the new layout and asked you how many employees do you need to test the new layout? You found out the the effect size for the same concept in the pilot study was 0.71. What would you tell him?

Let's assume this is an A/B test comparing the current portal layout with the proposed new version, I would recommend doing a power analysis before deploying it broadly.

Using the usual assumptions:
Effect size: Cohen's d = 0.71
Significance level alpha = 0.05
Statistical power: 80%
Two-sided test
Two roughly equal groups

The required sample is approximately 32 employees total, or about 16 employees per group.

In practice, I would recommend testing around 36-40 employees to allow for incomplete sessions, unusable observations, or droupouts.

We need roughly 32 employees statistically, but I would recruit about 40 employees--20 using the current layout and 20 using the new layout-to esure we have enough usable data.

Because the company has 1,000 employees, you do  not need to test the new interface on all 1,000. The relatively large effect size of 0.71 means the expected difference is substantial enough that a comparatively small randommized sample can detect it.

An experimental structure would be:

Employees -> random assignment -> Current Layout vs. New Layout -> compare performance

Additionally, I could then measure outcomes such as average call-handling time, ticket completion time, errors, nnumber of tickets completed, and user satisfaction.


In [2]:
from statsmodels.stats.power import TTestIndPower
import math

# Given values
effect_size = 0.71
alpha = 0.05
power = 0.80

# Create the power analysis object
analysis = TTestIndPower()

# Calculate required sample size per group
sample_size = analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative="two-sided"
)

# Round up because we cannot test part of an employee
sample_size_per_group = math.ceil(sample_size)

# Total emplyees needed
total_sample_size = sample_size_per_group * 2

print("Employees needed per group", sample_size_per_group)
print("Total employees needed:", total_sample_size)



Employees needed per group 33
Total employees needed: 66


In [3]:
import pandas as pd
import numpy as np

# Example employee IDS
employees = pd.DataFrame({
    "user_id": range(1, 1001)
})

# Randomly select 66 employees
sample = employees.sample(
    n=66,
    random_state=42
).copy()

# Randomly assign 33 to each Layout
sample["layout"] = (
    ["Current"] * 33 +
    ["New"] * 33
)

# Shuffle the assigments
sample["layout"] = np.random.default_rng(42).permutation(
    sample["layout"].values
)

print(sample.head())
print(sample["layout"].value_counts())

     user_id   layout
521      522      New
737      738  Current
740      741      New
660      661  Current
411      412      New
layout
New        33
Current    33
Name: count, dtype: int64


In [4]:
import math

required_sample = 66
dropout_rate = 0.10

adjusted_sample = math.ceil(required_sample / (1 - dropout_rate))

print("Adjusted sample size:", adjusted_sample)

Adjusted sample size: 74


In [5]:
import pandas as pd
import numpy as np

# Example list of 1,000 employee IDs
employees = pd.DataFrame({
    "user_id": range(1, 1001)
})

# Select 74 employees to allow for attrition
sample = employees.sample(n=74, random_state=42).copy()

# Randomly assign treatment groups
sample["layout"] = np.random.default_rng(42).choice(
    ["Current Layout", "New Layout"],
    size=len(sample)
)

print(sample.head())
print(sample["layout"].value_counts())

     user_id          layout
521      522  Current Layout
737      738      New Layout
740      741      New Layout
660      661  Current Layout
411      412  Current Layout
layout
New Layout        38
Current Layout    36
Name: count, dtype: int64


In [6]:
sample["layout"] = (
    ["Current Layout"] * 37 +
    ["New Layout"] * 37
)

sample["layout"] = np.random.default_rng(42).permutation(
    sample["layout"]
)

In [7]:
results = pd.DataFrame({
    "user_id": [1, 2, 3, 4],
    "layout": ["Current", "Current", "New", "New"],
    "avg_handle_time": [420, 390, 350, 340],
    "tickets_completed": [15, 17, 20, 22],
    "error_rate": [0.08, 0.06, 0.04, 0.03]
})

print(results)

   user_id   layout  avg_handle_time  tickets_completed  error_rate
0        1  Current              420                 15        0.08
1        2  Current              390                 17        0.06
2        3      New              350                 20        0.04
3        4      New              340                 22        0.03


In [8]:
from scipy.stats import ttest_ind

current = results[
    results["layout"] == "Current"
]["avg_handle_time"]

new = results[
    results["layout"] == "New"
]["avg_handle_time"]

t_stat, p_value = ttest_ind(
    current,
    new,
    equal_var=False
)

print("t-statistic:", t_stat)
print("p-value:", p_value)

t-statistic: 3.794733192202055
p-value: 0.1281348755918152


In [9]:
import numpy as np

def cohens_d(group1, group2):
  n1 = len(group1)
  n2 = len(group2)

  pooled_sd = np.sqrt(
      (
          (n1 -1) * np.var(group1, ddof=1)
          + (n2 -1) * np.var(group2, ddof=1)
      )
      / (n1 + n2 -2)
  )

  return (np.mean(group1) - np.mean(group2)) / pooled_sd

d = cohens_d(current, new)

print("Cohen's d:", d)

Cohen's d: 3.794733192202055


#Question 2: The boss then asked which employees would you need specifically so that the data team can pull their data for you? He needs a file consisting their user ids where you would also specify which users would have the old version and which users would they deploy the new versions to.

###show how to extract files from your analysis

In [10]:
#randomization
#print the list, export the list into a csv.

import pandas as pd
import numpy as np

# Load the employee IDS
# Example file: user_id.csv
users = pd.read_csv("user_id.csv")

# Set a seed so the selection can be reproduced
random_seed = 42

# Randomly select 66 employees from the 1,000
selected_users = users.sample(
    n=66,
    random_state=random_seed
).copy()

# Create balanced experimen groups
groups = (
    ["Old Version"] * 33 +
    ["New Version"] * 33
)

# Randomly shuffle the group assignments
rng = np.random.default_rng(random_seed)
selected_users["version"] = rng.permutation(groups)

# Check the allocation
print(selected_users["version"].value_counts())

# Save the file for the data/deployment team
selected_users.to_csv(
    "portal_layout_experiment_users.csv",
    index=False
)

print(selected_users.head())

FileNotFoundError: [Errno 2] No such file or directory: 'user_id.csv'

In [ ]:
employees.to_csv('user_id.csv', index=False)

In [ ]:
selected_users["treatment"] = selected_users["version"].map({
    "Old Version": 0,
    "New Version": 1
})

selected_users.to_csv(
    "portal_layout_ab_test.csv",
    index=False
)

In [ ]:
users = pd.read_csv("user_id.csv")

N_PER_GROUP = 33
TOTAL_SAMPLE = N_PER_GROUP * 2
SEED = 42

# Randomly sample employees
experiment = users.sample(
    n=TOTAL_SAMPLE,
    random_state=SEED
).copy()

# Create exactly equal groups
assignments = (
    ["Control"] * N_PER_GROUP +
    ["Treatment"] * N_PER_GROUP
)

# Randomize assignments
rng = np.random.default_rng(SEED)
experiment["experiment_group"] = rng.permutation(assignments)

# Identify which interface each group receives
experiment["portal_version"] = experiment["experiment_group"].map({
    "Control": "Old Version",
    "Treatment": "New Version"
})

# Sort by user ID if desired
experiment = experiment.sort_values("user_id")

# Save the deployment list
experiment.to_csv(
  "portal_ab_test_user_assignment.csv",
  index=False
)

print(experiment)
print()
print(experiment["experiment_group"].value_counts())

#PART 2 (FOR HOMEWORK/OPTIONAL)

use lab15_data.csv and perform all of the experiment steps you think you would need.



*Note: this is a different dataset*

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sample

from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.power import TTestIndPower

#==============================================
# 1. Load DeprecationWarning
#==============================================

df = pd.read_csv("lab15_data.csv")

print(df.head())
print(df.shape)
print(df.info())

# =============================================
# 2. Data Quality Checks
# =============================================

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate user IDs:")
print(df["user_id"].duplicate().sum())

print("\nGroup sizes:")
print(df["new_user_activation"].value_counts())

print("\nActivation values:")
print(df["new_user_activation"].value_counts())

print("\nPurchase summary:")
print(df["purchases"].describe())


# ===========================================
# 3. Descriptive Statistics
# ===========================================

summary = df.groupby("version").agg(
    employees=("user_id", "count"),
    activations=("new_user_activation", "sum")
    activation_rate=("new_user_activation", "mean"),
    avg_purchases=("purchases", "mean"),
    median_purchases=("purchases", "median"),
    sd_purchases=("purchases", "std"),
    total_purchases=("purchases", "sum")
)

print("\nGroup summary:")
print(summary)

# ===========================================
# 4. Activation A/B Test
# ===========================================

table = pd.crosstab(
    df["version"],
    df["new_user_activation"]
)

print("\nActivation contingency table:")
print(table)

a_success = table.loc["A", True]
b_success = table.loc["B", True]

a_n = table.loc["A"].sum()
a_b = table.loc["B"].sum()

counts = np.array([
    b_success,
    a_success
])

observations = np.array([
    b_n,
    a_n
])

z_stat, p_value = proportions_ztest(
    counts,
    observations
)

print("\nTwo-proportion z-test")
print("Z statistic:", z_stat)
print("P-value:", p_value)

rate_a = a_success / a_n
rate_b = b_success / b_n

difference = rate_b - rate_a

print("A activation:", rate_a)
print("B activation:", rate_b)
print("Absolute difference:", difference)
print("Relative lift:", rate_b / rate_a -1)

# ==================================================
# 6. Relative Risk and Odds Ratio
# ==================================================

relative_risk = rate_b / rate_a

odds_a = rate_a / (1-rate_a)
odds_b = rate_b / (1-rate_b)

odds_ratio = odds_b / odds_a

print("\nRelative risk:", relative_risk)
print("Odds ratio:", odds_ratio)

# ================================================
# 7. Purchase Data
# ================================================

group_a = df.loc[
    df["version"] == "A",
    "purchases"
]

print("\nMean purchases A:", group_a.mean())
print("Mean purchases B:", group_b.mean())

purchase_difference = (
    group_b.mean() -
    group_a.mean()
)

print("Difference:", purchase_difference)

# ================================================
# 8. Normality Check
# ================================================

print("\nShapiro test A:")
print(stats.shapiro(group_a))

print("\nShapiro test B:")
print(stats.shapiro(group_b))

# ================================================
# 9 Variance Check
# ================================================


levene = stats.levene(
    group_a,
    group_b,
    center = "median"
)

print("\nLevene test:")
print(levene)

# ================================================
# 10. Welch T-test
# ================================================

welch = stats.ttest_ind(
    group_b,
    group_a,
    equal_var=False
)

print("\nWelch t-test:")
print(welch)


# ===============================================
# 11. Mann-Whitney U Test
# ===============================================

mann_whitney = stats.mannwhitneyu(
    group_b,
    group_a,
    alternative="two-sided"
)

print("\nMann-Whitney test:")
print(mann_whitney)

# ===============================================
# 12. Cohen's D
# ===============================================

n_a = len(group_a)
n_b = len(group_b)

sd_a = group_a.std()
sd_b = group_b.std()

pooled_sd = np.sqrt(
    (
        (n_a -1) * sd_a**2
        + (n_b -1) * sd_b**2
    )
    /
    (n_a + n_b -2)
)

cohens_d = (
    group_b.mean() -
    group_a.mean()
) / pooled_sd

print("\nCohen's d:", cohens_d)


# =============================================
# 13. Poisson Regression
# =============================================

X = pd.get_dummies(
    df["version"],
    drop_first=True,
    dtype=int
)

X = sm.add_constant(X)

poisson = sm.GLM(
    df["purchases"],
    X,
    family=sm.families.Poisson()
).fit()

print("\nPoisson regression:")
print(poisson.summary())

irr = np.exp(poisson.params["B"])

print("\nIncidence Rate Ratio:")
print(irr)


# =================================================
# 14. Randomization Check Using User ID
# =================================================

a_ids = df.loc[
    df["version"] == "A",
    "user_id"
]

b_ids = df.loc[
    df["version"] == "B",
    "user_id"
]

randomization_test = stats.ttest_ind(
    a_ids,
    binequal_var=False
)

print("\nUser-ID balance check:")
print(randomization_test)

# ===========================================
# 15. Bootstrap
# ===========================================

rng = np.random.defalt_rng(42)

bootstrap_activation = []
bootstrap_purchases = []

a_df = df[df["version"] == "A"]
b_df = df[df["version"] == "B"]

for i in range(10000):

  sample_a = a_df.sample(
      len(a_df),
      replace=True,
      random_state=rng.integers(1_000_000_)
  )

  sample_b = b_df.sample(
      len(b_df),
      replace=True,
      random_state=rng.integers(1_000_000_)
  )

bootstrap_activation.append(
    sample_b["new_user_activation"].mean()
    -
    sample_a["new_user_activation"].mean()
)

bootstrap_purchases.append(
    sample_b["purchases"].mean()
    -
    sample_a["purchases"].mean()
)

print("\nActivation bootstrap CI:")
print(
    np.percentile(
        bootstrap_activation,
        [2.5, 97.5]
    )
)

print("\nPurchases bootstrap CI:")
print(
    np.percentile(
        bootstrap_purchases,
        [2.5, 97.5]
    )
)

# ========================================
# 16. Power Analysis
# ========================================

power_analysis = TTestIndPower()

power = power_analysis.power(
    effect_size=.71,
    nobs1=n_a,
    ratio=n_b/n_a,
    alpha=0.05
)

minimum_detectable_effec = (
    power_analysis.solve_power(
        effect_size=None,
        nobs1=n_a,
        ratio=n_b/n_a,
        alpha=0.05,
        power=0.8
    )
)

print("\nMinimum detectable Cohen's d:")
print(minimum_detectable_effect)

The A/B test had 1,000 users total, with 486 using Version A and 514 using Version B. The data looked clean—there were no missing values or duplicate user IDs.

Version B had a slightly better activation rate at 58.75%, compared with 55.97% for Version A. That is an increase of about 2.8 percentage points. However, the difference was not statistically significant, so we cannot confidently say Version B actually caused the improvement.

For purchases, Version B did a little worse. Users on Version B averaged 0.975 purchases, compared with 1.043 for Version A. We checked this using several different statistical tests, and none of them found a meaningful difference between the two versions. The effect size was also extremely small, which suggests the difference probably does not matter much in practical terms.

Overall, the results do not give us enough evidence to say Version B is better than Version A. There is a small positive signal for activation, but we would need more testing before recommending a full rollout.

### Bottom line

**Version B is not a clear winner.**

* **Activation:** B was about 2.8 percentage points higher, but the difference was not statistically significant.
* **Purchases:** B averaged about 0.07 fewer purchases per user, but again, the difference was not significant.
* **Effect size:** Very small, meaning the practical difference between A and B was minimal.
* **Pilot result:** The earlier effect size of 0.71 did not show up in this experiment.
* **Recommendation:** I would not roll out Version B based on these results alone. Before running another test, I would clearly define the main business metric we care about and how much improvement would actually be large enough to justify making the change.
